In [1]:
from libraries import *
from parameters import *
from util import *

import os, tempfile, uuid
import numpy as np
import pandas as pd
import scipy.sparse as sp
from joblib import Parallel, delayed
from scipy.spatial.distance import cdist
from statsmodels.stats.multitest import multipletests


/home/beraslan/miniconda/envs/py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
a=sc.read_h5ad("/processed_datasets/VCI/ChemoGenetic_H1_Basak/AllControls.h5ad")

In [13]:
a.layers

Layers with keys: 

In [8]:
a_round2=a[a.obs["sample"].isin(drugConditions_round2+drugConditions_round2_batch2),:]

In [9]:
a_round2.obs["sample"].value_counts()

sample
AR-A014418               20000
AZD4573                  20000
Bisindolylmaleimide-I    20000
CHIR-98014               20000
DG-172                   20000
DMSO_round2              20000
DMSO_round2_batch2       20000
JTE-607                  20000
LDN-193189               20000
LY2090314                20000
Lexibulin                20000
NSC95397                 20000
PP121                    20000
Romidepsin               20000
Stattic                  20000
VX-11e                   20000
Name: count, dtype: int64

In [10]:
a_round2.write("/processed_datasets/VCI/ChemoGenetic_H1_Basak/AllControls_scaled_round2Only.h5ad")

In [ ]:
a=sc.read_h5ad("/processed_datasets/VCI/ChemoGenetic_H1_Basak/Bisindolylmaleimide-I/SPLIT/Bisindolylmaleimide-I_controls.h5ad")

In [ ]:
k=pd.DataFrame(a.obs.assignment_mixture.value_counts())

In [ ]:
k

In [ ]:
k=k.loc[k["count"]>10]

In [ ]:
k

In [ ]:
a.write("/processed_datasets/VCI/ChemoGenetic_H1_Basak/DMSO_round2_batch2/DMSO_round2_batch2.h5ad")

In [ ]:
_MEMMAP = None
_SHAPE = None
_DTYPE = None

def _init_worker(mm_path, shape, dtype):
    """Open shared memmap read-only inside each worker."""
    global _MEMMAP, _SHAPE, _DTYPE
    _SHAPE = tuple(shape)
    _DTYPE = np.dtype(dtype)
    _MEMMAP = np.memmap(mm_path, mode='r', dtype=_DTYPE, shape=_SHAPE)



In [ ]:
def _chunked_mean_cdist(A, B, batch=100):
    nA, nB = A.shape[0], B.shape[0]
    total = 0.0
    count = 0
    for i in range(0, nA, batch):
        Ai = A[i:i+batch]
        for j in range(0, nB, batch):
            Bj = B[j:j+batch]
            D = cdist(Ai, Bj, metric="euclidean")
            total += D.sum()
            count += D.size
    return total / count if count else np.nan

def _chunked_mean_within(A, batch=100):
    n = A.shape[0]
    if n < 2:
        return 0.0
    total = 0.0
    for i0 in range(0, n, batch):
        Ai = A[i0:i0+batch]
        Dii = cdist(Ai, Ai, metric="euclidean")
        total += Dii[np.triu_indices_from(Dii, k=1)].sum()
        for j0 in range(i0+batch, n, batch):
            Aj = A[j0:j0+batch]
            Dij = cdist(Ai, Aj, metric="euclidean")
            total += Dij.sum()
    return (2.0 * total) / (n * (n - 1))

def energy_distance_full(A, B, batch=100):
    cross = _chunked_mean_cdist(A, B, batch=batch)
    within_A = _chunked_mean_within(A, batch=batch)
    within_B = _chunked_mean_within(B, batch=batch)
    D2 = 2.0 * cross - within_A - within_B
    return float(np.sqrt(max(D2, 0.0)))


In [ ]:
def _compute_for_pert(pert, pert_idx, ctrl_pool_idx, batch, rng_seed,
                      size_key, null_bank):
    
    rng = np.random.default_rng(rng_seed)
    nA, nB = size_key

    if len(pert_idx) > nA:
        pert_idx = rng.choice(pert_idx, size=nA, replace=False)
    ctrl_use = rng.choice(ctrl_pool_idx, size=nB, replace=False)

    A = np.asarray(_MEMMAP[pert_idx])
    B = np.asarray(_MEMMAP[ctrl_use])

    ed_obs = energy_distance_full(A, B, batch=batch)

    null_vals = null_bank[size_key]  
    ge = int((null_vals >= ed_obs).sum())
    n_null = null_vals.size
    pval = (ge + 1.0) / (n_null + 1.0)

    return {
        "perturbation": pert,
        "n_pert": nA,
        "n_ctrl": nB,
        "energy_distance": ed_obs,
        "null_mean": float(null_vals.mean()),
        "null_std": float(null_vals.std(ddof=1)) if n_null > 1 else 0.0,
        "pval": pval,
    }


In [ ]:
def parallel_energy_distance_with_shared_null(
    adata,
    pert_col="target_gene",
    control_label="non_targeting",
    dtype="float32",
    min_cells=20,
    n_null=200,              # draws per unique size pair
    batch=100,
    n_jobs=8,
    random_state=42,
    tmpdir="/processed_datasets/VCI/ChemoGenetic_H1_Basak/TmpFiles/"):
   
    # if sp.issparse(adata.X):
    #     M = adata.X.toarray().astype(dtype, copy=False)
    # else:
    #     M = np.asarray(adata.X, dtype=dtype)

    M = adata.obsm["X_pca"]
    
    mm_path = os.path.join(tmpdir, f"memmap_{uuid.uuid4().hex}.dat")
    mm = np.memmap(mm_path, mode='w+', dtype=M.dtype, shape=M.shape)
    mm[:] = M[:]
    del mm
    del M

    obs = adata.obs[pert_col].astype(str).values
    all_perts = np.array(sorted(set(obs) - {control_label}))
    idx_ctrl = np.where(obs == control_label)[0]
   
    rng = np.random.default_rng(random_state)
    
    # per-pert indices with caps applied
    tasks = []
    size_keys = []  # collect (nA, nB) needed
    for p in all_perts:
        idx = np.where(obs == p)[0]
        if idx.size < min_cells:
            continue
        
        nA = idx.size
        nB = min(idx_ctrl.size, nA)
        
        tasks.append((p, idx))
        size_keys.append((nA, nB))

    if not tasks:
        os.remove(mm_path)
        return pd.DataFrame(columns=[
            "perturbation","n_pert","n_ctrl","energy_distance",
            "null_mean","null_std","pval","qval"
        ])

    size_keys_unique = sorted(set(size_keys))
    null_bank = {}  
    _init_worker(mm_path, shape=adata.obsm["X_pca"].shape, dtype=dtype)
    parent_mm = np.memmap(mm_path, mode='r', dtype=np.dtype(dtype))
    shape =adata.obsm["X_pca"].shape
    parent_mm = np.memmap(mm_path, mode='r', dtype=np.dtype(dtype), shape=shape)

    for (nA, nB) in size_keys_unique:
        vals = np.empty(n_null, dtype="float32")
        for r in range(n_null):
            A_idx = rng.choice(idx_ctrl, size=nA, replace=False)
            remaining = np.setdiff1d(idx_ctrl, A_idx, assume_unique=False)
            if remaining.size < nB:
                B_idx = rng.choice(idx_ctrl, size=nB, replace=True)
            else:
                B_idx = rng.choice(remaining, size=nB, replace=False)
            A = np.asarray(parent_mm[A_idx])
            B = np.asarray(parent_mm[B_idx])
            vals[r] = energy_distance_full(A, B, batch=batch)
        null_bank[(nA, nB)] = vals

    # ---- run per-perturbation in parallel (reusing the shared null) ----
    seeds = rng.integers(0, 2**32 - 1, size=len(tasks), dtype=np.uint64)

    results = Parallel(n_jobs=n_jobs, backend="loky", verbose=0,
                       initializer=_init_worker, initargs=(mm_path, shape, dtype))(
        delayed(_compute_for_pert)(
            pert=t[0],
            pert_idx=t[1],
            ctrl_pool_idx=idx_ctrl,
            batch=batch,
            rng_seed=int(seeds[i]),
            size_key=size_keys[i],
            null_bank=null_bank
        )
        for i, t in enumerate(tasks)
    )

    # cleanup
    try:
        os.remove(mm_path)
    except Exception:
        pass

    ed_df = pd.DataFrame(results)
    if ed_df.empty:
        return ed_df
    ed_df["qval"] = multipletests(ed_df["pval"], alpha=0.05, method="fdr_bh")
    ed_df = ed_df.sort_values(["pval", "energy_distance"], ascending=[True, False]).reset_index(drop=True)
    return ed_df


In [ ]:
cond="CHIR"
adata=sc.read_h5ad("/processed_datasets/VCI/ChemoGenetic_H1_Basak/"+cond+"/SPLIT/ad_001.h5ad")

In [ ]:
adata_sel = adata[adata.obs["target_gene"].isin(["non-targeting","BAX","AKAP12"]),]

In [ ]:
sc.pp.scale(adata_sel, max_value=15)
sc.pp.pca(adata_sel, n_comps=100, svd_solver='arpack')


In [ ]:
adata_sel.obsm["X_pca"]

In [ ]:
pert_col="target_gene"
control_label="non-targeting"
dtype="float32"
min_cells=20
n_null=200             # draws per unique size pair
batch=100
n_jobs=8
random_state=42
tmpdir="./TempDir"

In [ ]:
if sp.issparse(adata.X):
    M = adata.X.toarray().astype(dtype, copy=False)
else:
    M = np.asarray(adata.X, dtype=dtype)
    
mm_path = os.path.join(tmpdir, f"memmap_{uuid.uuid4().hex}.dat")
mm = np.memmap(mm_path, mode='w+', dtype=M.dtype, shape=M.shape)
mm[:] = M[:]
del mm
del M


In [ ]:
obs = adata.obs[pert_col].astype(str).values
all_perts = np.array(sorted(set(obs) - {control_label}))
idx_ctrl = np.where(obs == control_label)[0]

rng = np.random.default_rng(random_state)

# per-pert indices with caps applied
tasks = []
size_keys = []  # collect (nA, nB) needed
for p in all_perts:
    idx = np.where(obs == p)[0]
    if idx.size < min_cells:
        continue
    
    nA = idx.size
    nB = min(idx_ctrl.size, nA)
    
    tasks.append((p, idx))
    size_keys.append((nA, nB))


In [ ]:
obs

In [ ]:
k = parallel_energy_distance_with_shared_null(
    adata=adata_sel,
    pert_col="target_gene",
    control_label="non-targeting",
    dtype="float32",
    min_cells=20,
    n_null=200,              # draws per unique size pair
    batch=100,
    n_jobs=8,
    random_state=42,
    tmpdir="/processed_datasets/VCI/ChemoGenetic_H1_Basak/TmpFiles/")